# CephFS Shared Storage

This notebook demonstrates how to use the **`storage`** flag to automatically mount a CephFS shared filesystem on your FABRIC nodes. With a single flag, FABlib will:

1. Add a FABNetv4 network to the node (required for Ceph connectivity)
2. Generate your Ceph credentials
3. Upload them to the node
4. Mount CephFS at `/mnt/cephfs/<cluster>/<user>/`

This notebook covers three usage patterns:
- **Slice-level**: All nodes get CephFS storage
- **Node-level**: Only selected nodes get CephFS storage
- **Explicit cluster**: Specify which Ceph cluster to use

### Prerequisites

- Your project must have CephFS storage allocated. Contact your project lead or FABRIC admins.
- Complete the [Configure Environment](../../../configure_and_validate.ipynb) notebook first.

### FABlib API References

- [slice.add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) — `storage` and `storage_cluster` parameters
- [slice.submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) — `storage` and `storage_cluster` parameters
- [node.enable_storage](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.enable_storage)
- [node.has_storage](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.has_storage)

## Setup: Import FABlib

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

fablib.show_config();

---
## Pattern 1: Slice-Level Storage (All Nodes)

The simplest approach — pass `storage=True` to `slice.submit()` and **every** node in the slice gets CephFS mounted automatically.

This is ideal when all nodes need access to the same shared data.

In [ ]:
slice = fablib.new_slice(name="CephFS-AllNodes")

node1 = slice.add_node(name="node1", site="TACC")
node2 = slice.add_node(name="node2", site="STAR")

# All nodes get CephFS storage
slice.submit(storage=True);

### Verify the mounts

CephFS is mounted under `/mnt/cephfs/<cluster>/<user>/`. Let's confirm on both nodes.

In [ ]:
for node in slice.get_nodes():
    print(f"--- {node.get_name()} ---")
    stdout, stderr = node.execute("df -h | grep cephfs")

### Write from one node, read from another

Since CephFS is a shared filesystem, data written on one node is immediately visible on the other.

In [ ]:
# Find the CephFS mount point on node1
node1 = slice.get_node("node1")
node2 = slice.get_node("node2")

# Discover the mount path
stdout, stderr = node1.execute("mount | grep cephfs | awk '{print $3}' | head -1", quiet=True)
mount_path = stdout.strip()
print(f"CephFS mount: {mount_path}")

# Write a file from node1
stdout, stderr = node1.execute(f"echo 'Hello from node1!' > {mount_path}/test_file.txt")

# Read it from node2
stdout, stderr = node2.execute(f"cat {mount_path}/test_file.txt")

### Clean up Pattern 1

In [ ]:
slice.delete()

---
## Pattern 2: Node-Level Storage (Selective)

Pass `storage=True` to `add_node()` to enable CephFS on specific nodes only. Nodes without the flag will not get a CephFS mount.

This is useful when only some nodes need storage access (e.g., a compute node reads data, a control node doesn't need it).

In [ ]:
slice = fablib.new_slice(name="CephFS-Selective")

# This node gets CephFS
data_node = slice.add_node(name="data-node", site="TACC", storage=True)

# This node does NOT get CephFS
control_node = slice.add_node(name="control-node", site="TACC")

slice.submit();

### Verify selective mounting

In [ ]:
for node in slice.get_nodes():
    print(f"--- {node.get_name()} (storage={node.has_storage()}) ---")
    stdout, stderr = node.execute("df -h | grep cephfs || echo 'No CephFS mount'")

### Clean up Pattern 2

In [ ]:
slice.delete()

---
## Pattern 3: Explicit Ceph Cluster

By default, the first available Ceph cluster is auto-discovered. If your project has access to multiple clusters, you can specify which one to use with the `storage_cluster` parameter.

### Discover available clusters

In [ ]:
clusters = fablib.discover_ceph_clusters()
print(f"Available Ceph clusters: {clusters}")

### Create a slice targeting a specific cluster

In [ ]:
# Use the first discovered cluster (replace with your preferred cluster)
cluster_name = clusters[0]
print(f"Using cluster: {cluster_name}")

slice = fablib.new_slice(name="CephFS-ExplicitCluster")

node = slice.add_node(name="node1", site="TACC", storage=True, storage_cluster=cluster_name)

slice.submit();

### Verify the mount and inspect storage details

In [ ]:
node = slice.get_node("node1")

print(f"Storage enabled: {node.has_storage()}")
print(f"Storage cluster: {node.get_storage_cluster()}")
print()

stdout, stderr = node.execute("df -h | grep cephfs")
stdout, stderr = node.execute("ls -la /mnt/cephfs/")

### Clean up Pattern 3

In [ ]:
slice.delete()